In [50]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [51]:
Dataset = pd.read_csv("dataset.csv")
print("Data base loaded: ",Dataset.shape)

Data base loaded:  (35975, 18)


In [52]:
Dataset = Dataset.replace(r'^\s*$', np.nan, regex=True)
Dataset = Dataset.drop_duplicates()
Dataset = Dataset.dropna()
Dataset = Dataset[Dataset["type"]!="mitm"]
Dataset = Dataset.drop(Dataset.columns[0], axis=1)
print("Database cleaned: ",Dataset.shape)

Database cleaned:  (35949, 17)


In [53]:
x = Dataset.drop("type",axis=1)
y = Dataset["type"]

In [56]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

num_classes = len(np.unique(y_encoded))

x_temp,x_test,y_temp,y_test =train_test_split(x,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded) #80%(training + validation) 20% test
x_train,x_val,y_train,y_val =train_test_split(x_temp,y_temp,test_size=0.25,random_state=42,stratify=y_temp) #60% training 20% validation

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train) #here we calculating the mean and ecartype (fit_) and then z-score (transform)
x_val_scaled = scaler.transform(x_val)       #so we dont have data leakage we should use the same mean and ecart so we did just (transform)
x_test_scaled = scaler.transform(x_test)     #same goes here to avoid data leakage

smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train_scaled, y_train)

print("the models entry X and Y succefully generated")

the models entry X and Y succefully generated


In [57]:
model = Sequential()

model.add(Dense(64,activation='relu',input_shape=(x_train_scaled.shape[1],)))

model.add(Dropout(0.2))

model.add(Dense(32,activation='relu'))

model.add(Dense(num_classes,activation='softmax'))

model.summary()

c:\Users\Ouail\Documents\Projects\projet_stage\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,399 (13.28 KB)

 Trainable params: 3,399 (13.28 KB)

 Non-trainable params: 0 (0.00 B)

In [58]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='PR-AUC')])

In [59]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

history = model.fit(
    x_train_resampled,y_train_resampled,
    epochs=100,
    batch_size = 32,
    validation_data=(x_val_scaled,y_val),
    verbose=2,
    callbacks=[early_stop],
)

Epoch 1/100
3264/3264 - 9s - 3ms/step - PR-AUC: 0.8368 - loss: 0.4869 - val_PR-AUC: 0.7771 - val_loss: 0.5619
Epoch 2/100
3264/3264 - 7s - 2ms/step - PR-AUC: 0.9239 - loss: 0.2397 - val_PR-AUC: 0.8499 - val_loss: 0.4067
Epoch 3/100
3264/3264 - 7s - 2ms/step - PR-AUC: 0.9453 - loss: 0.1803 - val_PR-AUC: 0.8839 - val_loss: 0.3470
Epoch 4/100
3264/3264 - 7s - 2ms/step - PR-AUC: 0.9556 - loss: 0.1520 - val_PR-AUC: 0.8880 - val_loss: 0.3421
Epoch 5/100
3264/3264 - 7s - 2ms/step - PR-AUC: 0.9611 - loss: 0.1369 - val_PR-AUC: 0.8968 - val_loss: 0.3380
Epoch 6/100
3264/3264 - 7s - 2ms/step - PR-AUC: 0.9641 - loss: 0.1255 - val_PR-AUC: 0.8990 - val_loss: 0.3408
Epoch 7/100
3264/3264 - 7s - 2ms/step - PR-AUC: 0.9665 - loss: 0.1194 - val_PR-AUC: 0.9079 - val_loss: 0.2788
Epoch 8/100
3264/3264 - 8s - 2ms/step - PR-AUC: 0.9681 - loss: 0.1144 - val_PR-AUC: 0.9099 - val_loss: 0.3021
Epoch 9/100
3264/3264 - 8s - 2ms/step - PR-AUC: 0.9696 - loss: 0.1088 - val_PR-AUC: 0.9070 - val_loss: 0.2932
Epoch 10/1

In [60]:
# Get general accuracy
loss, accuracy = model.evaluate(x_test_scaled, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

# Make predictions (these will be probabilities due to softmax)
y_pred_probs = model.predict(x_test_scaled)

# Convert probabilities to actual class predictions (choose the highest probability)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Print a detailed classification report
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - PR-AUC: 0.9317 - loss: 0.2261
Test Accuracy: 0.9317
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
              precision    recall  f1-score   support

        ddos       0.94      0.98      0.96       922
         dos       0.41      0.97      0.58       105
   injection       0.67      0.93      0.78       122
      normal       0.99      0.91      0.95      4972
    password       0.97      0.98      0.98       726
    scanning       0.53      0.96      0.69        89
         xss       0.68      0.98      0.81       254

    accuracy                           0.93      7190
   macro avg       0.74      0.96      0.82      7190
weighted avg       0.95      0.93      0.94      7190



In [61]:
import joblib

# 1. Save the Keras neural network
model.save('ids_mlp_model.keras')

# 2. Save the fitted StandardScaler
joblib.dump(scaler, 'scaler.joblib')

# 3. Save the LabelEncoder
joblib.dump(label_encoder, 'label_encoder.joblib')

print("All artifacts successfully saved!")

All artifacts successfully saved!
